# 01 - EDA and Cleaning
Initial data exploration and cleaning steps.

In [3]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix, csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, Tuple, Optional
import ast
import gc # Garbage Collector để dọn rác RAM chủ động
import pickle
from pathlib import Path
from scipy import sparse
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.exceptions import NotFittedError


# CollaborativeFeatureEngineer Class

In [4]:
class CollaborativeFeatureEngineer:
    """
    Module xử lý Feature Engineering cho hệ thống gợi ý sử dụng ma trận thưa.
    Tối ưu hóa cho dữ liệu lớn và bộ nhớ RAM hạn chế.
    """

    def __init__(self):
        self.user_to_idx: Dict = {}
        self.idx_to_user: Dict = {}
        self.movie_to_idx: Dict = {}
        self.idx_to_movie: Dict = {}
        self.user_means: Optional[np.ndarray] = None
        self.item_means: Optional[np.ndarray] = None
        self.matrix_normalized: Optional[csr_matrix] = None

    def fit_transform(self, df: pd.DataFrame) -> csr_matrix:
        """
        Thực hiện toàn bộ quy trình từ mapping đến chuẩn hóa.
        
        Args:
            df: DataFrame chứa columns ['userId', 'movieId', 'rating']
            
        Returns:
            csr_matrix: Ma trận đã được chuẩn hóa (Double-centered).
        """
        # Bước 1: Mapping IDs
        row_indices, col_indices = self._create_mappings(df)
        
        # Bước 2: Build Sparse Matrix
        # Sử dụng float32 để tiết kiệm 50% RAM so với float64
        sparse_mat = self._build_sparse_matrix(
            df,
            row_indices=row_indices,
            col_indices=col_indices,
        )

        # Bước 3: Double-Centering
        self.matrix_normalized = self._apply_double_centering(sparse_mat)
        
        return self.matrix_normalized

    def transform(self, df: pd.DataFrame) -> csr_matrix:
        """
        Biến đổi dữ liệu mới bằng mapping và bias đã học từ tập train.

        Các user/item chưa xuất hiện trong lúc fit sẽ bị loại bỏ.
        """
        if self.matrix_normalized is None or self.user_means is None or self.item_means is None:
            raise NotFittedError("CollaborativeFeatureEngineer chưa được fit. Gọi fit_transform() trước.")

        sparse_mat = self._build_sparse_matrix(df, use_existing_mappings=True)
        if sparse_mat.nnz == 0:
            return sparse_mat

        mat = sparse_mat.copy()

        # Trừ bias user theo đúng thứ tự các hàng trong CSR matrix
        row_counts = np.diff(mat.indptr)
        mat.data -= np.repeat(self.user_means, row_counts)

        # Trừ bias item theo đúng thứ tự các cột sau khi chuyển sang CSC
        mat_csc = mat.tocsc()
        col_counts = np.diff(mat_csc.indptr)
        mat_csc.data -= np.repeat(self.item_means, col_counts)

        return mat_csc.tocsr()

    def _build_sparse_matrix(
        self,
        df: pd.DataFrame,
        row_indices: Optional[np.ndarray] = None,
        col_indices: Optional[np.ndarray] = None,
        use_existing_mappings: bool = False,
    ) -> csr_matrix:
        """Dựng sparse matrix từ DataFrame và mapping hiện có."""
        if use_existing_mappings:
            if not self.user_to_idx or not self.movie_to_idx:
                raise NotFittedError("CollaborativeFeatureEngineer chưa có mapping. Gọi fit_transform() trước.")

            row_mapped = df['userId'].map(self.user_to_idx)
            col_mapped = df['movieId'].map(self.movie_to_idx)
            valid_mask = row_mapped.notna() & col_mapped.notna()

            if not valid_mask.any():
                return csr_matrix((len(self.user_to_idx), len(self.movie_to_idx)), dtype=np.float32)

            row_indices = row_mapped[valid_mask].astype(np.int32).values
            col_indices = col_mapped[valid_mask].astype(np.int32).values
            ratings = df.loc[valid_mask, 'rating'].values.astype(np.float32)
        else:
            if row_indices is None or col_indices is None:
                row_indices, col_indices = self._create_mappings(df)
            ratings = df['rating'].values.astype(np.float32)

        return coo_matrix(
            (ratings, (row_indices, col_indices)),
            shape=(len(self.user_to_idx), len(self.movie_to_idx))
        ).tocsr()

    def _create_mappings(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Tạo từ điển ánh xạ hai chiều và trả về mảng index."""
        unique_users = df['userId'].unique()
        unique_movies = df['movieId'].unique()

        self.user_to_idx = {uid: i for i, uid in enumerate(unique_users)}
        self.idx_to_user = {i: uid for uid, i in self.user_to_idx.items()}
        
        self.movie_to_idx = {mid: i for i, mid in enumerate(unique_movies)}
        self.idx_to_movie = {i: mid for mid, i in self.movie_to_idx.items()}

        row_indices = df['userId'].map(self.user_to_idx).values.astype(np.int32)
        col_indices = df['movieId'].map(self.movie_to_idx).values.astype(np.int32)
        
        return row_indices, col_indices

    def _apply_double_centering(self, matrix: csr_matrix) -> csr_matrix:
        """Khử bias User và Item trên cấu trúc thưa."""
        # Chế độ tính toán tránh biến đổi ma trận thưa thành ma trận dày (dense)
        mat = matrix.copy()
        
        # 1. Row-wise mean centering (User bias)
        # Chỉ tính trung bình trên các phần tử khác 0
        row_sums = mat.sum(axis=1).A1
        row_counts = np.diff(mat.indptr)
        self.user_means = np.divide(row_sums, row_counts,out=np.zeros_like(row_sums, dtype=np.float32), where=row_counts > 0)
        
        # Trừ trung bình hàng
        mat.data -= np.repeat(self.user_means, row_counts)

        # 2. Column-wise mean centering (Item bias)
        mat_csc = mat.tocsc()
        col_sums = mat_csc.sum(axis=0).A1
        col_counts = np.diff(mat_csc.indptr)
        self.item_means = np.divide(col_sums, col_counts,out=np.zeros_like(col_sums, dtype=np.float32), where=col_counts > 0)
        
        # Trừ trung bình cột
        mat_csc.data -= np.repeat(self.item_means, col_counts)
        
        return mat_csc.tocsr()

    def get_sparsity(self) -> float:
        """Tính toán phần trăm độ thưa của ma trận."""
        if self.matrix_normalized is None:
            return 0.0
        n_elements = self.matrix_normalized.shape[0] * self.matrix_normalized.shape[1]
        n_nonzero = self.matrix_normalized.nnz
        sparsity = (1 - n_nonzero / n_elements) * 100
        return sparsity

    def visualize_distribution(self, original_ratings: pd.Series):
        """Vẽ biểu đồ phân phối trước và sau chuẩn hóa."""
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        sns.histplot(original_ratings, bins=10, kde=True, color='blue')
        plt.title("Original Rating Distribution")
        
        plt.subplot(1, 2, 2)
        sns.histplot(self.matrix_normalized.data, bins=50, kde=True, color='red')
        plt.title("Double-Centered Rating Distribution")
        plt.show()

    def plot_top_interactions_heatmap(self, k: int = 50):
        """Vẽ Heatmap cho Top-K User và Item có nhiều tương tác nhất."""
        # Tìm top users/items dựa trên số lượng tương tác (nnz)
        row_counts = np.diff(self.matrix_normalized.indptr)
        top_users = np.argsort(row_counts)[-k:]
        
        col_counts = np.diff(self.matrix_normalized.tocsc().indptr)
        top_items = np.argsort(col_counts)[-k:]
        
        # Trích xuất ma trận con (Sub-matrix)
        sub_mat = self.matrix_normalized[top_users, :][:, top_items].toarray()
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(sub_mat, cmap='RdBu_r', center=0)
        plt.title(f"Heatmap of Top {k}x{k} Interactions (Normalized)")
        plt.xlabel("Movie Index")
        plt.ylabel("User Index")
        plt.show()

    def get_meta_data(self) -> Dict:
        """Xuất dữ liệu mapping và các giá trị bias."""
        return {
            "user_mapping": self.user_to_idx,
            "movie_mapping": self.movie_to_idx,
            "user_means": self.user_means,
            "item_means": self.item_means
        }

# ContentBasedRecommender Class

In [ ]:
class ContentBasedRecommender:
    """
    Hệ thống gợi ý dựa trên nội dung sử dụng TF-IDF Vectorization.
    Tối ưu hóa để cấu trúc tương tự CollaborativeFeatureEngineer.
    """
    
    def __init__(self, max_features: int = 10000, min_df: int = 5, max_df: float = 0.8):
        """
        Khởi tạo ContentBasedRecommender.
        
        Args:
            max_features (int): Số lượng từ vựng tối đa.
            min_df (int): Cắt bỏ các từ xuất hiện dưới ngưỡng này.
            max_df (float): Cắt bỏ các từ xuất hiện trên tỷ lệ này.
        """
        self._download_nltk_resources()
        
        # Lưu cấu hình
        self.max_features = max_features
        self.min_df = min_df
        self.max_df = max_df
        
        # Khởi tạo các thành phần xử lý text
        self.stop_words: set = set(stopwords.words('english'))
        self.lemmatizer: WordNetLemmatizer = WordNetLemmatizer()
        
        # Vectorizer (chưa fit)
        self.vectorizer: Optional[TfidfVectorizer] = None
        self.tfidf_matrix: Optional[csr_matrix] = None
        self.movie_to_idx: Dict = {}

    def _create_mapping(self, df: pd.DataFrame, feature_col: str) -> None:
        """Tạo mapping từ movieId gốc sang index cho ma trận TF-IDF."""
        unique_movies = df['id'].unique()
        self.movie_to_idx = {mid: i for i, mid in enumerate(unique_movies)}
        if len(self.movie_to_idx) != 0:
            print(f"Đã tạo mapping cho {len(self.movie_to_idx)} movies.")
        else:
            Warning("movie_to_idx mapping is empty after creation.")

    def _download_nltk_resources(self) -> None:
        """Tải các tài nguyên NLTK cần thiết một cách an toàn."""
        resources = ['stopwords', 'wordnet', 'omw-1.4']
        for res in resources:
            try:
                nltk.data.find(f'corpora/{res}')
            except LookupError:
                nltk.download(res, quiet=True)

    def _preprocess_text_series(self, text_series: pd.Series) -> pd.Series:
        """
        Tiền xử lý dữ liệu văn bản (Vectorized operations).
        
        Quy trình: NaN handling -> Lowercase -> Xóa ký tự đặc biệt 
                   -> Lemmatize & Stopwords removal.
        
        Args:
            text_series: Series chứa text cần xử lý.
            
        Returns:
            Processed Series.
        """
        # 1. Xử lý null và ép kiểu
        cleaned = text_series.fillna("").astype(str)
        
        # 2. Lowercase (vectorized)
        cleaned = cleaned.str.lower()
        
        # 3. Xóa ký tự đặc biệt (vectorized regex)
        cleaned = cleaned.str.replace(r'[^a-z0-9\s]', ' ', regex=True)
        
        # 4. Lemmatize & Stopwords removal (apply với pre-bound methods)
        lemmatize = self.lemmatizer.lemmatize
        stopwords_set = self.stop_words
        
        def process_tokens(text: str) -> str:
            return " ".join([
                lemmatize(word) for word in text.split()
                if word not in stopwords_set and len(word) > 1
            ])
        
        cleaned = cleaned.apply(process_tokens)
        return cleaned

    def fit_transform(self, df: pd.DataFrame, feature_col: str) -> csr_matrix:
        """
        Huấn luyện vectorizer và biến đổi text corpus thành TF-IDF matrix.
        
        Args:
            df (pd.DataFrame): DataFrame chứa dữ liệu.
            feature_col (str): Tên cột chứa text gộp.
            
        Returns:
            csr_matrix: Ma trận thưa TF-IDF đã được chuẩn hóa.
        """
        # Validation
        if feature_col not in df.columns:
            raise ValueError(f"Cột '{feature_col}' không tồn tại trong DataFrame.")
        
        # Bước 1: Tiền xử lý văn bản
        print("Bước 1: Tiền xử lý văn bản...")
        processed_text = self._preprocess_text_series(df[feature_col])
        
        # Bước 2: Khởi tạo và fit vectorizer
        print("Bước 2: Xây dựng TF-IDF matrix...")
        self.vectorizer = TfidfVectorizer(
            max_features=self.max_features,
            min_df=self.min_df,
            max_df=self.max_df,
            stop_words='english',
            dtype=np.float32  # RAM optimization
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(processed_text)
        
        vocab_size = len(self.vectorizer.vocabulary_)
        print(f"Hoàn thành!")
        print(f"  - Kích thước từ vựng: {vocab_size}")
        print(f"  - Kích thước matrix: {self.tfidf_matrix.shape}")
        print(f"  - Số phần tử khác 0: {self.tfidf_matrix.nnz:,}")
        

        self._create_mapping(df, feature_col)
        return self.tfidf_matrix

    def transform(self, df: pd.DataFrame, feature_col: str) -> csr_matrix:
        """
        Biến đổi dữ liệu mới bằng vectorizer đã fit trên tập train.

        Args:
            df (pd.DataFrame): DataFrame cần biến đổi.
            feature_col (str): Tên cột chứa text gộp.

        Returns:
            csr_matrix: TF-IDF matrix của tập mới với cùng số chiều như train.
        """
        if self.vectorizer is None:
            raise NotFittedError("Vectorizer chưa được huấn luyện. Gọi fit_transform() trước.")

        if feature_col not in df.columns:
            raise ValueError(f"Cột '{feature_col}' không tồn tại trong DataFrame.")

        processed_text = self._preprocess_text_series(df[feature_col])
        return self.vectorizer.transform(processed_text)

    def get_sparsity(self) -> float:
        """
        Tính độ thưa của ma trận TF-IDF.
        
        Returns:
            float: Phần trăm độ thưa (0-100).
        """
        if self.tfidf_matrix is None:
            return 0.0
        
        n_elements = self.tfidf_matrix.shape[0] * self.tfidf_matrix.shape[1]
        n_nonzero = self.tfidf_matrix.nnz
        sparsity = (1 - n_nonzero / n_elements) * 100
        return sparsity

    def get_feature_names(self) -> np.ndarray:
        """
        Lấy danh sách từ vựng đã được học.
        
        Returns:
            np.ndarray: Array chứa tên các features.
            
        Raises:
            NotFittedError: Nếu vectorizer chưa được fit.
        """
        if self.vectorizer is None:
            raise NotFittedError("Vectorizer chưa được huấn luyện. Gọi fit_transform() trước.")
        
        try:
            return self.vectorizer.get_feature_names_out()
        except AttributeError:
            raise NotFittedError("Vectorizer không hỗ trợ get_feature_names_out()")

    def get_meta_data(self) -> Dict:
        """
        Xuất metadata của mô hình đã fit.
        
        Returns:
            Dict: Chứa vectorizer, vocabulary, idf weights.
        """
        if self.vectorizer is None:
            raise NotFittedError("Mô hình chưa được fit. Gọi fit_transform() trước.")
        
        return {
            "vectorizer": self.vectorizer,
            "vocabulary": self.vectorizer.vocabulary_,
            "idf_weights": self.vectorizer.idf_,
            "feature_names": self.get_feature_names(),
            "matrix_shape": self.tfidf_matrix.shape,
            "sparsity_percent": self.get_sparsity(),
            "movie_mapping": self.movie_to_idx
        }

# Hướng dẫn tích hợp:
# recommender = ContentBasedRecommender(max_features=15000, min_df=5, max_df=0.8)
# tfidf_matrix = recommender.fit_transform(df, 'content_feature')
# metadata = recommender.get_meta_data()

# ETL

In [19]:
class MovieDataPipeline:
    def __init__(self, metadata_path, ratings_path, keywords_path=None, credits_path=None):
        """
        Khởi tạo Pipeline với đường dẫn file.
        """
        self.metadata_path = metadata_path
        self.ratings_path = ratings_path
        self.keywords_path = keywords_path
        self.credits_path = credits_path
        self.movies_df = None
        self.train_movies_df = None
        self.test_movies_df = None
        self.ratings_df = None
        self.train_ratings_df = None
        self.test_ratings_df = None
        self.cold_movies_df = None
        self.cold_ratings_df = None
        self.warm_movies_df = None
        self.warm_rating_df = None
        self.cold_movie_ids = None
        self.warm_movie_ids = None
        self.collab_matrix = None
        self.test_collab_matrix = None
        self.tfidf_matrix = None
        self.test_tfidf_matrix = None

    @staticmethod
    def _safe_literal_eval(value):
        if pd.isna(value):
            return []
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []

    @staticmethod
    def _extract_names(json_like_str):
        items = MovieDataPipeline._safe_literal_eval(json_like_str)
        return ' '.join(str(item.get('name', '')).strip() for item in items if item.get('name'))

    @staticmethod
    def _extract_credits_text(cast_str, crew_str):
        cast_items = MovieDataPipeline._safe_literal_eval(cast_str)
        crew_items = MovieDataPipeline._safe_literal_eval(crew_str)

        top_cast = [
            str(member.get('name', '')).strip()
            for member in cast_items[:5]
            if member.get('name')
        ]
        directors = [
            str(member.get('name', '')).strip()
            for member in crew_items
            if str(member.get('job', '')).lower() == 'director' and member.get('name')
        ]
        return ' '.join(top_cast + directors)

    def load_and_optimize(self):
        """
        Nạp dữ liệu kèm kỹ thuật Read-time Downcasting (Ép kiểu ngay khi đọc).
        Đây là kỹ thuật sống còn để chạy dữ liệu lớn trên laptop.
        """
        print("1. Đang nạp và tối ưu hóa bộ nhớ...")
        
        # Chỉ nạp những cột thật sự cần thiết. Bỏ qua các cột rác như homepage, tagline...
        metadata_cols = ['id', 'title', 'genres']
        
        # Đọc metadata, xử lý lỗi dòng (error_bad_lines) do file csv gốc thỉnh thoảng lỗi format
        self.movies_df = pd.read_csv(self.metadata_path, usecols=metadata_cols, low_memory=False)
        
        # Xử lý cột ID của movies: Loại bỏ các ID không phải là số (dữ liệu nhiễu) và ép về int32
        self.movies_df['id'] = pd.to_numeric(self.movies_df['id'], errors='coerce')
        self.movies_df = self.movies_df.dropna(subset=['id'])
        self.movies_df['id'] = self.movies_df['id'].astype('int32')

        # Nạp và tạo cột keywords từ keywords.csv
        if self.keywords_path is not None and Path(self.keywords_path).exists():
            keywords_df = pd.read_csv(self.keywords_path, usecols=['id', 'keywords'], low_memory=False)
            keywords_df['id'] = pd.to_numeric(keywords_df['id'], errors='coerce')
            keywords_df = keywords_df.dropna(subset=['id'])
            keywords_df['id'] = keywords_df['id'].astype('int32')
            keywords_df['keywords'] = keywords_df['keywords'].apply(self._extract_names)
            self.movies_df = self.movies_df.merge(keywords_df[['id', 'keywords']], on='id', how='left')
        else:
            self.movies_df['keywords'] = ''

        # Nạp và tạo cột credits từ credits.csv (gồm top cast + director)
        if self.credits_path is not None and Path(self.credits_path).exists():
            credits_df = pd.read_csv(self.credits_path, usecols=['id', 'cast', 'crew'], low_memory=False)
            credits_df['id'] = pd.to_numeric(credits_df['id'], errors='coerce')
            credits_df = credits_df.dropna(subset=['id'])
            credits_df['id'] = credits_df['id'].astype('int32')
            credits_df['credits'] = credits_df.apply(
                lambda row: self._extract_credits_text(row['cast'], row['crew']), axis=1
            )
            self.movies_df = self.movies_df.merge(credits_df[['id', 'credits']], on='id', how='left')
        else:
            self.movies_df['credits'] = ''

        self.movies_df['keywords'] = self.movies_df['keywords'].fillna('')
        self.movies_df['credits'] = self.movies_df['credits'].fillna('')

        # Đọc Ratings: Chỉ định trước Data Types (Kỹ thuật ăn tiền với dân C++)
        # Thay vì tốn 64 bits cho rating, ta dùng float32. user_id và movie_id dùng int32
        rating_dtypes = {
            'userId': 'int32',
            'movieId': 'int32',
            'rating': 'float32'
        }
        # Chỉ lấy 3 cột, bỏ qua 'timestamp' vì RecSys cơ bản chưa dùng tới Context-aware
        self.ratings_df = pd.read_csv(self.ratings_path, usecols=['userId', 'movieId', 'rating'], dtype=rating_dtypes)
        
        print(f"-> Đã nạp thành công! Ratings shape: {self.ratings_df.shape}")

    def clean_metadata(self):
        """
        Xử lý Missing Values và làm sạch Text cho Content-Based.
        """
        print("2. Đang làm sạch Metadata...")
        
        # Đảm bảo title luôn hợp lệ cho downstream hiển thị/recommendation.
        self.movies_df = self.movies_df.dropna(subset=['title'])
        
        # Cột genres trong dataset này đang ở dạng String của List of Dictionaries (VD: "[{'id': 12, 'name': 'Animation'}]")
        # Ta cần parse string này thành Python object bằng ast.literal_eval, sau đó rút trích tên thể loại.
        def extract_genres(x):
            try:
                # Trích xuất 'name' từ list các dict
                genres = ast.literal_eval(x)
                return ' '.join([i['name'] for i in genres])
            except:
                return ''
                
        self.movies_df['genres_text'] = self.movies_df['genres'].apply(extract_genres)
        self.movies_df = self.movies_df.drop(columns=['genres']) # Xóa cột cũ giải phóng RAM

    def prune_interactions(self, min_movie_ratings=10, min_user_ratings=5):
        """
        Cắt tỉa ma trận User-Item. Loại bỏ Users vãng lai và Phim quá chìm.
        """
        print(f"3. Đang cắt tỉa ma trận (Lọc User >= {min_user_ratings} rates, Movie >= {min_movie_ratings} rates)...")
        
        # Bước 1: Giữ các phim có từ min_movie_ratings đánh giá trở lên
        movie_counts = self.ratings_df['movieId'].value_counts()
        valid_movies = movie_counts[movie_counts >= min_movie_ratings].index
        self.ratings_df = self.ratings_df[self.ratings_df['movieId'].isin(valid_movies)]
        
        # Bước 2: Giữ các user có từ min_user_ratings đánh giá trở lên
        user_counts = self.ratings_df['userId'].value_counts()
        valid_users = user_counts[user_counts >= min_user_ratings].index
        self.ratings_df = self.ratings_df[self.ratings_df['userId'].isin(valid_users)]
        
        # Chỉ định lại index để dọn dẹp RAM dư thừa từ Pandas View
        self.ratings_df = self.ratings_df.reset_index(drop=True)
        gc.collect() # Ép Garbage Collector chạy tay để dọn dẹp biến tạm
        
        print(f"-> Sau cắt tỉa: Ratings shape thu gọn còn {self.ratings_df.shape}")

    def extract_features(self):
        """
        Chuẩn bị Feature cuối cùng cho 2 nhánh mô hình.
        """
        print("4. Chuẩn bị đặc trưng (Feature Extraction)...")
        
        # Đảm bảo dữ liệu Ratings chỉ tham chiếu tới các Phim có tồn tại trong Metadata
        valid_metadata_ids = self.movies_df['id'].unique()
        self.ratings_df = self.ratings_df[self.ratings_df['movieId'].isin(valid_metadata_ids)]
        
        # Content-Based Feature: Gộp Text
        # Content-based models (như TF-IDF hoặc Word2Vec) cần 1 chuỗi văn bản tổng hợp.
        self.movies_df['content_feature'] = (
            self.movies_df['title'] + " " + self.movies_df['genres_text'] + " " + self.movies_df['keywords'] + " " + self.movies_df['credits']
        )
        
    def split_movies_train_test(self, test_size: float = 0.2, random_state: int = 42, shuffle: bool = True):
        """
        Chia movies_df thành train/test để fit/transform TF-IDF theo cùng tỷ lệ.

        Args:
            test_size (float): Tỷ lệ tập test, mặc định 0.2.
            random_state (int): Seed để tái lập kết quả.
            shuffle (bool): Có xáo trộn dữ liệu trước khi chia hay không.

        Returns:
            Tuple[pd.DataFrame, pd.DataFrame]: (train_movies_df, test_movies_df)
        """
        if self.movies_df is None or self.movies_df.empty:
            raise ValueError('movies_df chưa sẵn sàng. Hãy chạy load_and_optimize() và clean_metadata() trước.')

        train_df, test_df = train_test_split(
            self.movies_df,
            test_size=test_size,
            random_state=random_state,
            shuffle=shuffle,
        )

        self.train_movies_df = train_df.reset_index(drop=True)
        self.test_movies_df = test_df.reset_index(drop=True)

        print(f"5. Đã tách movies_df: train={self.train_movies_df.shape}, test={self.test_movies_df.shape}")
        return self.train_movies_df, self.test_movies_df

    def split_train_test(self, test_size: float = 0.2, random_state: int = 42, shuffle: bool = True):
        """
        Chia ratings thành train/test theo tỷ lệ mong muốn.

        Args:
            test_size (float): Tỷ lệ tập test, mặc định 0.2.
            random_state (int): Seed để tái lập kết quả.
            shuffle (bool): Có xáo trộn dữ liệu trước khi chia hay không.

        Returns:
            Tuple[pd.DataFrame, pd.DataFrame]: (train_ratings_df, test_ratings_df)
        """
        if self.ratings_df is None or self.ratings_df.empty:
            raise ValueError('ratings_df chưa sẵn sàng. Hãy chạy load_and_optimize() và prune_interactions() trước.')

        train_df, test_df = train_test_split(
            self.warm_rating_df,
            test_size=test_size,
            random_state=random_state,
            shuffle=shuffle,
        )

        self.train_warm_ratings_df = train_df.reset_index(drop=True)
        self.test_warm_ratings_df = test_df.reset_index(drop=True)

        print(f"6. Đã tách dữ liệu ratings: train={self.train_warm_ratings_df.shape}, test={self.test_warm_ratings_df.shape}")
        return self.train_warm_ratings_df, self.test_warm_ratings_df

    def split_train_test_cold_warm(self, cold_item_ratio: float = 0.05, random_state: int = 42):
        """
        Tách dữ liệu thành 2 nhóm theo movieId:
        1) Cold group (cold_item_ratio): cold_movies_df, cold_ratings_df.
        2) Warm group còn lại: warm_movies_df, warm_rating_df.

        Returns:
            Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
                (warm_movies_df, warm_rating_df, cold_movies_df, cold_ratings_df)
        """
        if self.ratings_df is None or self.ratings_df.empty:
            raise ValueError('ratings_df chưa sẵn sàng. Hãy chạy load_and_optimize() và prune_interactions() trước.')

        if not (0 < cold_item_ratio < 1):
            raise ValueError('cold_item_ratio phải nằm trong khoảng (0, 1).')

        rng = np.random.RandomState(random_state)
        unique_movies = self.ratings_df['movieId'].dropna().unique()

        if unique_movies.size < 2:
            raise ValueError('Cần ít nhất 2 movieId để thực hiện split cold/warm.')

        # Giai đoạn 1: Chọn cold items và rút toàn bộ interactions của chúng ra cold_ratings_df.
        n_cold = int(np.floor(unique_movies.size * cold_item_ratio))
        n_cold = max(1, n_cold)
        n_cold = min(n_cold, unique_movies.size - 1)

        cold_movie_ids = rng.choice(unique_movies, size=n_cold, replace=False)
        cold_mask = self.ratings_df['movieId'].isin(cold_movie_ids)

        cold_ratings_df = self.ratings_df.loc[cold_mask].reset_index(drop=True)
        warm_pool_df = self.ratings_df.loc[~cold_mask].reset_index(drop=True)

        if warm_pool_df.empty:
            raise ValueError('Warm pool rỗng sau khi tách cold items. Hãy giảm cold_item_ratio.')

        if self.movies_df is not None and not self.movies_df.empty:
            self.cold_movies_df = self.movies_df[self.movies_df['id'].isin(cold_movie_ids)].reset_index(drop=True)
            self.warm_movies_df = self.movies_df[self.movies_df['id'].isin(warm_pool_df['movieId'].unique())].reset_index(drop=True)
        else:
            self.cold_movies_df = pd.DataFrame()
            self.warm_movies_df = pd.DataFrame()

        self.warm_rating_df = warm_pool_df
        self.cold_ratings_df = cold_ratings_df
        self.cold_movie_ids = np.array(sorted(set(cold_movie_ids)))
        self.warm_movie_ids = np.array(sorted(set(warm_pool_df['movieId'].unique())))

        print(f"6a. Cold-start split: {len(self.cold_movie_ids)} movieId ({len(self.cold_movie_ids)/len(unique_movies):.2%})")
        print(f"    -> cold_movies_df shape: {self.cold_movies_df.shape}")
        print(f"    -> cold_ratings_df shape: {self.cold_ratings_df.shape}")
        print(f"6b. Warm group: warm_movies_df={self.warm_movies_df.shape}, warm_rating_df={self.warm_rating_df.shape}")
        print(f"    -> warm_movies_df shape: {self.warm_movies_df.shape}")
        print(f"    -> warm_rating_df shape: {self.warm_rating_df.shape}")
        print(f"    -> warm movieId count: {len(self.warm_movie_ids)}")

        return self.warm_movies_df, self.warm_rating_df, self.cold_movies_df, self.cold_ratings_df

    def show_statistics(self):
        """
        In ra Thống kê mô tả (Descriptive Statistics).
        """
        print("\n=== THỐNG KÊ MÔ TẢ TỔNG QUAN ===")
        print("1. Phân phối điểm đánh giá (Ratings Distribution):")
        print(self.ratings_df['rating'].describe())
        
        sparsity = 1.0 - (len(self.ratings_df) / (self.ratings_df['userId'].nunique() * self.ratings_df['movieId'].nunique()))
        print(f"\n2. Độ thưa của ma trận (Sparsity): {sparsity * 100:.4f}%")
        print("   (Điều này có nghĩa là {:.4f}% ô trong ma trận đang bị trống - Rất bình thường trong RecSys!)".format(sparsity * 100))

    def build_collaborative_artifacts(self, df, output_dir, file_prefix='ratings', engineer=None):
        """
        Dùng CollaborativeFeatureEngineer để tạo ma trận thưa chuẩn hóa cho 1 DataFrame truyền vào
        và lưu artifacts (.npz và .pkl) phục vụ huấn luyện/inference.
        
        Args:
            df: DataFrame cần xử lý.
            output_dir: Thư mục lưu file.
            file_prefix: Tiền tố tên file (ví dụ 'train', 'test').
            engineer: Object CollaborativeFeatureEngineer đã được fit (dành cho tập test).
                    Nếu None, sẽ tự động tạo mới và fit_transform (dành cho tập train).
        """
        print(f"5. Đang tối ưu dữ liệu Collaborative cho '{file_prefix}'...")

        if df is None or df.empty:
            raise ValueError(f'Không có dữ liệu (df trống) để xây dựng artifacts cho {file_prefix}.')

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        matrix_path = output_dir / f'{file_prefix}_matrix.npz'
        meta_path = output_dir / f'{file_prefix}_mapping.pkl'

        # Phân nhánh xử lý cho tập Train (chưa có engineer) và Test (đã có engineer)
        if engineer is None:
            engineer = CollaborativeFeatureEngineer()
            collab_matrix = engineer.fit_transform(df)
            meta_data = engineer.get_meta_data()
            sparsity = engineer.get_sparsity()
        else:
            collab_matrix = engineer.transform(df)
            meta_data = dict(engineer.get_meta_data())
            meta_data['split'] = file_prefix
            # Nếu class của bạn có hàm tính sparsity cho dữ liệu transform, gọi ở đây. 
            # Tạm thời gán bằng 'N/A' hoặc có thể tự tính.
            sparsity = 'N/A (Chỉ tính trên tập train)' 

        # Lưu file ma trận thưa (.npz)
        sparse.save_npz(matrix_path, collab_matrix)
        
        # Lưu file metadata (.pkl)
        with open(meta_path, 'wb') as f:
            pickle.dump(meta_data, f)

        # In log
        print(f"-> Matrix shape: {collab_matrix.shape}, nnz={collab_matrix.nnz}")
        if isinstance(sparsity, float):
            print(f"-> Sparsity sau chuẩn hóa: {sparsity:.4f}%")
        print(f"-> Đã lưu artifacts:\n- Matrix: {matrix_path}\n- Metadata: {meta_path}\n")

        return collab_matrix, engineer

    def build_tfidf_artifacts(self, df, output_dir, feature_col='content_feature', file_prefix='movies', recommender=None, max_features=10000, min_df=1, max_df=0.8):
        """
        Dùng ContentBasedRecommender để tạo ma trận TF-IDF cho 1 DataFrame truyền vào
        và lưu artifacts (.npz, .pkl) phục vụ huấn luyện/inference.

        Args:
            df: DataFrame đầu vào chứa cột text cần vector hóa.
            output_dir: Thư mục đầu ra chứa artifacts.
            feature_col: Tên cột text để trích xuất TF-IDF.
            file_prefix: Tiền tố tên file (ví dụ 'movies_train', 'movies_test').
            recommender: Instance ContentBasedRecommender đã fit (dành cho tập test/inference).
                        Nếu None, sẽ khởi tạo mới và fit_transform (dành cho tập train).
            max_features, min_df, max_df: Cấu hình cho ContentBasedRecommender.

        Returns:
            tfidf_matrix: Ma trận TF-IDF dạng thưa (scipy.sparse).
            recommender: Instance ContentBasedRecommender.
        """
        print(f"7. Đang xây dựng TF-IDF artifacts cho '{file_prefix}'...")

        if df is None or df.empty:
            raise ValueError(f"Không có dữ liệu (df trống) để xây dựng TF-IDF artifacts cho {file_prefix}.")

        if feature_col not in df.columns:
            raise ValueError(f"Cột '{feature_col}' không tồn tại trong DataFrame đầu vào.")

        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        matrix_path = output_dir / f'{file_prefix}_matrix.npz'
        meta_path = output_dir / f'{file_prefix}_mapping.pkl'

        # Phân nhánh: Train (fit mới) hoặc Test/Inference (chỉ transform)
        if recommender is None:
            recommender = ContentBasedRecommender(max_features=max_features, min_df=min_df, max_df=max_df)
            tfidf_matrix = recommender.fit_transform(df, feature_col=feature_col)
            meta_data = recommender.get_meta_data()
        else:
            tfidf_matrix = recommender.transform(df, feature_col=feature_col)
            meta_data = dict(recommender.get_meta_data())
            meta_data['split'] = file_prefix

        # Lưu ma trận TF-IDF thưa bằng thư viện SciPy
        sparse.save_npz(matrix_path, tfidf_matrix)
        
        # Lưu metadata mapping
        with open(meta_path, 'wb') as f:
            pickle.dump(meta_data, f)

        print(f"-> TF-IDF Matrix shape: {tfidf_matrix.shape}, nnz={tfidf_matrix.nnz}")
        print(f"-> Đã lưu artifacts:\n- Matrix: {matrix_path}\n- Metadata: {meta_path}\n")

        return tfidf_matrix, recommender

# --- CÁCH SỬ DỤNG PIPELINE ---
if __name__ == "__main__":
    # Tự động suy ra root project để chạy được từ mọi thư mục hiện hành
    project_root = next(
        (path for path in [Path.cwd(), *Path.cwd().parents]
         if (path / "data" / "raw").exists() and (path / "data" / "processed").exists()),
        None,
)

    if project_root is None:
        raise FileNotFoundError("Không tìm thấy cấu trúc data/raw và data/processed trong thư mục hiện tại hoặc các thư mục cha.")

    METADATA_PATH = project_root / 'data' / 'raw' / 'movies_metadata.csv'
    RATINGS_PATH = project_root / 'data' / 'raw' / 'ratings.csv'  # Có thể thử với ratings_small.csv trước để test logic
    KEYWORDS_PATH = project_root / 'data' / 'raw' / 'keywords.csv'
    CREDITS_PATH = project_root / 'data' / 'raw' / 'credits.csv'
    processed_dir = project_root / 'data' / 'processed'
    processed_dir.mkdir(parents=True, exist_ok=True)
    
    # Khởi tạo và chạy tuần tự
    pipeline = MovieDataPipeline(METADATA_PATH, RATINGS_PATH, KEYWORDS_PATH, CREDITS_PATH)
    pipeline.load_and_optimize()
    pipeline.clean_metadata()
    pipeline.prune_interactions(min_movie_ratings=10, min_user_ratings=5)
    pipeline.extract_features()
    pipeline.split_train_test_cold_warm(cold_item_ratio=0.05, random_state=42)
    pipeline.split_train_test(test_size=0.2, random_state=42)
    pipeline.show_statistics()
    # Mẹo nhỏ: Lưu lại dưới dạng parquet (nhanh hơn, lưu trữ cấu trúc type int32/float32 tốt hơn CSV)
    warm_ratings_path = processed_dir / 'warm_ratings.parquet'
    cold_ratings_path = processed_dir / 'cold_ratings.parquet'
    train_warm_ratings_path = processed_dir / 'train_warm_ratings.parquet'
    test_warm_ratings_path = processed_dir / 'test_warm_ratings.parquet'
    train_warm_movies_path = processed_dir / 'train_warm_movies.parquet'
    cold_movies_path = processed_dir / 'cold_movies.parquet'
    pipeline.train_warm_ratings_df.to_parquet(train_warm_ratings_path)
    pipeline.test_warm_ratings_df.to_parquet(test_warm_ratings_path)
    pipeline.cold_ratings_df.to_parquet(cold_ratings_path)
    pipeline.warm_movies_df.to_parquet(train_warm_movies_path)
    pipeline.cold_movies_df.to_parquet(cold_movies_path)
    # pipeline.train.to_parquet(train_movies_path)
    # pipeline.test_movies_df.to_parquet(test_movies_path)
    print(f"\n5. Đã lưu dữ liệu xử lý:")
    print(f"- Train warm ratings: {train_warm_ratings_path}")
    print(f"- Test warm ratings: {test_warm_ratings_path}")
    print(f"- Cold ratings: {cold_ratings_path}")
    print(f"- Train warm movies: {train_warm_movies_path}")
    print(f"- Cold movies: {cold_movies_path}")
    pipeline.build_collaborative_artifacts(df=pipeline.train_warm_ratings_df, output_dir=processed_dir,file_prefix='train_warm_ratings')
    pipeline.build_collaborative_artifacts(df=pipeline.test_warm_ratings_df, output_dir=processed_dir, engineer=pipeline.collab_matrix, file_prefix='test_warm_ratings')
    pipeline.build_collaborative_artifacts(df=pipeline.cold_ratings_df, output_dir=processed_dir, engineer=pipeline.collab_matrix, file_prefix='cold_ratings')
    pipeline.build_tfidf_artifacts(df=pipeline.warm_movies_df,feature_col='content_feature', output_dir=processed_dir, max_features=10000, min_df=5, max_df=0.8, file_prefix='train_warm_movies')
    pipeline.build_tfidf_artifacts(df=pipeline.cold_movies_df, feature_col='content_feature', output_dir=processed_dir, recommender=pipeline.tfidf_matrix, file_prefix='cold_movies')



1. Đang nạp và tối ưu hóa bộ nhớ...
-> Đã nạp thành công! Ratings shape: (26024289, 3)
2. Đang làm sạch Metadata...
3. Đang cắt tỉa ma trận (Lọc User >= 5 rates, Movie >= 10 rates)...
-> Sau cắt tỉa: Ratings shape thu gọn còn (25916033, 3)
4. Chuẩn bị đặc trưng (Feature Extraction)...
6a. Cold-start split: 267 movieId (4.99%)
    -> cold_movies_df shape: (269, 6)
    -> cold_ratings_df shape: (684649, 3)
6b. Warm group: warm_movies_df=(5151, 6), warm_rating_df=(10727701, 3)
    -> warm_movies_df shape: (5151, 6)
    -> warm_rating_df shape: (10727701, 3)
    -> warm movieId count: 5079
6. Đã tách dữ liệu ratings: train=(8582160, 3), test=(2145541, 3)

=== THỐNG KÊ MÔ TẢ TỔNG QUAN ===
1. Phân phối điểm đánh giá (Ratings Distribution):
count    1.141235e+07
mean     3.532923e+00
std      1.066535e+00
min      5.000000e-01
25%      3.000000e+00
50%      4.000000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

2. Độ thưa của ma trận (Sparsity): 99.1643%
   (Đi

# Summarize of Data 

ratings_train_matrix.npz được tối ưu từ train_ratings.parquet
ratings_test_matrix.npz được tối ưu từ test_ratings.parquet
movies_train_matrix.npz được tối ưu từ train_movies.parquet
movies_test_matrix.npz được tối ưu từ test_movies.parquet

tfidf_matrices được tối ưu từ train_movies.parquet và test_movies.parquet


Mô tả dữ liệu được xử lý:
movies_df: DataFrame chứa thông tin phim đã được làm sạch, với các cột như id, title, genres_text, keywords, credits, content_feature (chuỗi tổng hợp cho Content-Based)
ratings_df: DataFrame chứa thông tin đánh giá đã được cắt tỉa, với các cột userId, movieId, rating (đã ép kiểu int32/float32 để tiết kiệm RAM)
id là khóa chính để liên kết giữa movies_df và ratings_df. Các cột genres_text, keywords, credits đã được trích xuất và làm sạch để phục vụ cho mô hình Content-Based Filtering. Ma trận User-Item đã được cắt tỉa để loại bỏ các user vãng lai và phim quá chìm, giúp tăng hiệu quả của mô hình Collaborative Filtering.
genres_text: Chuỗi chứa tên các thể loại phim, được trích xuất từ cột genres gốc (dạng string của list of dicts).
keywords: Chuỗi chứa tên các từ khóa liên quan đến phim (Nội dung phim)
credits: Chuỗi chứa tên các diễn viên chính và đạo diễn (Thông tin về dàn diễn viên và đạo diễn)
content_feature: Chuỗi tổng hợp từ genres_text + keywords + credits, được sử dụng làm đặc trưng đầu vào cho mô hình Content-Based Filtering.
userId: ID của người dùng (đã ép kiểu int32)
movieId: ID của phim (đã ép kiểu int32)
rating: Điểm đánh giá của user cho movie (đã ép kiểu float32)
collab_matrix_normalized.npz: shape = (255451, 5346), nnz = 11412350
tfidf_matrix.npz: shape = (46625, 10000), nnz = 739181
Trong đó nnz là số phần tử khác 0 của sparse matrix.

In [12]:
pipeline.warm_movies_df.head()

,id,title,keywords,credits,genres_text
0,862,Toy Story,jealousy toy boy friendship friends rivalry bo...,Tom Hanks Tim Allen Don Rickles Jim Varney Wal...,Animation Comedy Family
1,8844,Jumanji,board game disappearance based on children's b...,Robin Williams Jonathan Hyde Kirsten Dunst Bra...,Adventure Fantasy Family
2,949,Heat,robbery detective bank obsession chase shootin...,Al Pacino Robert De Niro Val Kilmer Jon Voight...,Action Crime Drama Thriller
3,710,GoldenEye,cuba falsely accused secret identity computer ...,Pierce Brosnan Sean Bean Izabella Scorupco Fam...,Adventure Action Thriller
4,1408,Cutthroat Island,exotic island treasure map ship scalp pirate,Geena Davis Matthew Modine Frank Langella Maur...,Action Adventure


In [13]:
print(pipeline.movies_df.head())

      id                        title  \
0    862                    Toy Story   
1   8844                      Jumanji   
2  15602             Grumpier Old Men   
3  31357            Waiting to Exhale   
4  11862  Father of the Bride Part II   

                                            keywords  \
0  jealousy toy boy friendship friends rivalry bo...   
1  board game disappearance based on children's b...   
2   fishing best friend duringcreditsstinger old men   
3  based on novel interracial relationship single...   
4  baby midlife crisis confidence aging daughter ...   

                                             credits  \
0  Tom Hanks Tim Allen Don Rickles Jim Varney Wal...   
1  Robin Williams Jonathan Hyde Kirsten Dunst Bra...   
2  Walter Matthau Jack Lemmon Ann-Margret Sophia ...   
3  Whitney Houston Angela Bassett Loretta Devine ...   
4  Steve Martin Diane Keaton Martin Short Kimberl...   

                genres_text                                    content_feature 